In [1]:
import json
import re
import ast
import pandas as pd
from pathlib import Path
from collections import defaultdict

from utils import fix_mojibake, canonical_normalize, build_normalized_text_and_map

# -------------------------
# CONFIG
# -------------------------
ABSTRACTS_JSONL = "Dataset/VIOLIN_12-10-2025/interim/pubmed_abstracts.jsonl"
EXACT_JSONL = "Dataset/VIOLIN_12-10-2025/interim/exact_matches/matched_exact_abstracts.jsonl"
FUZZY_JSONL = "Dataset/VIOLIN_12-10-2025/interim/fuzzy_matches/matched_fuzzy_abstracts.jsonl"
PMIDS_ALL = "Dataset/VIOLIN_12-10-2025/interim/pmids_all.txt"
LEXICON_CSV = "Dataset/VIOLIN_12-10-2025/interim/adjuvant_ner_lexicon_clean.csv"

OUT_JSONL = "Dataset/VIOLIN_12-10-2025/final/llm_adjuvant_evidence_corpus.jsonl"
OUT_PRETTY = "Dataset/VIOLIN_12-10-2025/final/llm_adjuvant_evidence_corpus_pretty.json"

USE_EXACT = True
USE_FUZZY = True
FUZZY_MIN_SCORE = 96  # adjust if needed

# -------------------------
# Load infectious PMIDs
# -------------------------
infectious_pmids = set()
with open(PMIDS_ALL, encoding="utf-8") as f:
    for line in f:
        pmid = line.strip()
        if pmid:
            infectious_pmids.add(pmid)

# -------------------------
# PMID -> title/abstract map
# -------------------------
pmid_to_abs = {}
with open(ABSTRACTS_JSONL, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        pmid = str(rec.get("pmid", "")).strip()
        if pmid:
            pmid_to_abs[pmid] = {
                "title": fix_mojibake(rec.get("title", "")),
                "abstract": fix_mojibake(rec.get("abstract", "")),
            }

# -------------------------
# Lexicon display label map
# -------------------------
def is_better_label(cur, cand):
    if not cur:
        return True

    def score(s):
        s = s.strip()
        has_lower = any(c.islower() for c in s)
        has_space = " " in s
        return (1 if has_lower else 0, 1 if has_space else 0, len(s))

    return score(cand) > score(cur)

def to_list(x):
    if isinstance(x, list):
        return x
    if not isinstance(x, str):
        return []
    s = x.strip()
    if not s:
        return []
    try:
        if s.startswith("[") and s.endswith("]"):
            return ast.literal_eval(s)
    except Exception:
        pass
    return [s]

norm_to_display = {}

def consider_label(label):
    if not isinstance(label, str):
        return
    label = fix_mojibake(label).strip()
    if not label:
        return
    key = canonical_normalize(label)
    if not key:
        return
    cur = norm_to_display.get(key)
    if cur is None or is_better_label(cur, label):
        norm_to_display[key] = label

lex = pd.read_csv(LEXICON_CSV)

for _, row in lex.iterrows():
    consider_label(row.get("preferred_name", ""))

for col in ["synonyms", "expanded_forms"]:
    for _, row in lex.iterrows():
        for item in to_list(row.get(col)):
            consider_label(item)

# -------------------------
# Helpers
# -------------------------
SENT_DELIMS = ".!?\n"

def extract_sentence(raw_text, start, end):
    left = max(raw_text.rfind(d, 0, start) for d in SENT_DELIMS)
    right_candidates = [raw_text.find(d, end) for d in SENT_DELIMS if raw_text.find(d, end) != -1]
    right = min(right_candidates) if right_candidates else len(raw_text)
    sent_start = left + 1
    sent_end = right
    return raw_text[sent_start:sent_end].strip()

def make_regex(term):
    if not isinstance(term, str):
        return None
    term = canonical_normalize(term)
    if not term:
        return None
    pat = re.escape(term)
    pat = pat.replace(r"\ ", r"\s+")
    pat = pat.replace(r"\-", r"[-\s]+")
    return re.compile(rf"\b{pat}\b", flags=re.I)

# -------------------------
# Collect candidate terms
# -------------------------
pmid_to_terms = defaultdict(list)

def add_term(pmid, label, match_text, score):
    if not label or not match_text:
        return
    if not isinstance(label, str):
        label = str(label)
    if not isinstance(match_text, str):
        match_text = str(match_text)

    label = fix_mojibake(label.strip())
    match_text = fix_mojibake(match_text.strip())
    if not label or not match_text:
        return

    key = canonical_normalize(label)
    if key in norm_to_display:
        label = norm_to_display[key]

    pmid_to_terms[pmid].append((label, match_text, score))

if USE_EXACT:
    with open(EXACT_JSONL, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            pmid = str(rec.get("pmid", "")).strip()
            if not pmid:
                continue
            for m in rec.get("matches", []):
                add_term(pmid, m.get("matched_text", ""), m.get("matched_text", ""), 100)

if USE_FUZZY:
    with open(FUZZY_JSONL, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            pmid = str(rec.get("pmid", "")).strip()
            if not pmid:
                continue
            for m in rec.get("matches", []):
                score = m.get("score", 0)
                if score < FUZZY_MIN_SCORE:
                    continue
                label = m.get("dictionary_term", "") or m.get("matched_text", "")
                add_term(pmid, label, m.get("matched_text", ""), score)

# -------------------------
# Build corpus (best score per adjuvant)
# -------------------------
out_lines = []

for pmid, pairs_list in pmid_to_terms.items():
    if pmid not in infectious_pmids:
        continue

    abs_rec = pmid_to_abs.get(pmid)
    if not abs_rec:
        continue

    title_raw = abs_rec.get("title", "")
    abstract_raw = abs_rec.get("abstract", "")

    title_norm, title_map = build_normalized_text_and_map(title_raw)
    abstract_norm, abstract_map = build_normalized_text_and_map(abstract_raw)

    best = {}  # key -> {label, evidence, score}

    for label, match_text, score in pairs_list:
        regex = make_regex(match_text)
        if regex is None:
            continue

        for norm_text, raw_text, idx_map in [
            (title_norm, title_raw, title_map),
            (abstract_norm, abstract_raw, abstract_map),
        ]:
            if not norm_text:
                continue
            for m in regex.finditer(norm_text):
                start_raw = idx_map[m.start()] if m.start() < len(idx_map) else None
                end_raw = idx_map[m.end() - 1] + 1 if (m.end() - 1) < len(idx_map) else None
                if start_raw is None or end_raw is None:
                    continue

                evidence = extract_sentence(raw_text, start_raw, end_raw)
                if not evidence:
                    continue

                key = canonical_normalize(label) or label.lower()
                cur = best.get(key)
                if cur is None or score > cur["score"] or (score == cur["score"] and len(evidence) < len(cur["evidence"])):
                    best[key] = {"label": label, "evidence": evidence, "score": score}

    if not best:
        continue

    adjuvant_evidence = [
        {"adjuvant": v["label"], "evidence": v["evidence"], "confidence": v["score"]}
        for v in sorted(best.values(), key=lambda x: x["label"].lower())
    ]

    out_lines.append({
        "pmid": pmid,
        "title": title_raw,
        "abstract": abstract_raw,
        "adjuvant_evidence": adjuvant_evidence
    })

# -------------------------
# Save outputs
# -------------------------
Path(OUT_JSONL).parent.mkdir(parents=True, exist_ok=True)

with open(OUT_JSONL, "w", encoding="utf-8") as out:
    for rec in out_lines:
        out.write(json.dumps(rec, ensure_ascii=False) + "\n")

with open(OUT_PRETTY, "w", encoding="utf-8") as f:
    json.dump(out_lines, f, ensure_ascii=False, indent=2)

print(f"Saved: {OUT_JSONL}")
print(f"Saved: {OUT_PRETTY}")
print("Records:", len(out_lines))


Saved: Dataset/VIOLIN_12-10-2025/final/llm_adjuvant_evidence_corpus.jsonl
Saved: Dataset/VIOLIN_12-10-2025/final/llm_adjuvant_evidence_corpus_pretty.json
Records: 298


In [2]:
import json
from pathlib import Path

IN_JSONL = "Dataset/VIOLIN_12-10-2025/final/llm_adjuvant_evidence_corpus.jsonl"
OUT_JSONL = "Dataset/VIOLIN_12-10-2025/final/llm_adjuvant_evidence_corpus_finetune.jsonl"

def strip_confidence(adjuvant_evidence):
    cleaned = []
    for item in adjuvant_evidence:
        cleaned.append({
            "adjuvant": item.get("adjuvant", ""),
            "evidence": item.get("evidence", "")
        })
    return cleaned

with open(IN_JSONL, encoding="utf-8") as f_in, open(OUT_JSONL, "w", encoding="utf-8") as f_out:
    for line in f_in:
        rec = json.loads(line)
        finetune_rec = {
            "pmid": rec.get("pmid", ""),
            "input": f"Title: {rec.get('title','')}\nAbstract: {rec.get('abstract','')}",
            "output": {
                "adjuvant_evidence": strip_confidence(rec.get("adjuvant_evidence", []))
            }
        }
        f_out.write(json.dumps(finetune_rec, ensure_ascii=False) + "\n")

print(f"Saved: {OUT_JSONL}")


Saved: Dataset/VIOLIN_12-10-2025/final/llm_adjuvant_evidence_corpus_finetune.jsonl


In [3]:
import json
from pathlib import Path

IN_JSONL = "Dataset/VIOLIN_12-10-2025/final/llm_adjuvant_evidence_corpus_finetune.jsonl"
OUT_PRETTY = "Dataset/VIOLIN_12-10-2025/final/llm_adjuvant_evidence_corpus_finetune_pretty.json"

records = []
with open(IN_JSONL, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

Path(OUT_PRETTY).write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved: {OUT_PRETTY}")


Saved: Dataset/VIOLIN_12-10-2025/final/llm_adjuvant_evidence_corpus_finetune_pretty.json


In [1]:
"""import json
from pathlib import Path
from datasets import load_dataset
import random

DATA_JSONL = "Dataset/VIOLIN_12-10-2025/final/llm_adjuvant_evidence_corpus_finetune.jsonl"
OUT_JSON = "Dataset/VIOLIN_12-10-2025/final/splits_fixed.json"
TEST_RATIO = 0.10
SEED = 42

ds = load_dataset("json", data_files={"train": DATA_JSONL}, split="train")
pmids = sorted({r["pmid"] for r in ds})

rnd = random.Random(SEED)
rnd.shuffle(pmids)

n_test = max(1, int(len(pmids) * TEST_RATIO))
test_pmids = pmids[:n_test]
train_pmids = pmids[n_test:]

Path("results").mkdir(parents=True, exist_ok=True)
Path(OUT_JSON).write_text(json.dumps({
    "seed": SEED,
    "test_ratio": TEST_RATIO,
    "train_pmids": train_pmids,
    "test_pmids": test_pmids
}, indent=2), encoding="utf-8")

print("Saved:", OUT_JSON)
print("Train PMIDs:", len(train_pmids))
print("Test PMIDs:", len(test_pmids))
"""

Generating train split: 0 examples [00:00, ? examples/s]

Saved: Dataset/VIOLIN_12-10-2025/final/splits_fixed.json
Train PMIDs: 269
Test PMIDs: 29


In [1]:
#!/usr/bin/env python
# coding: utf-8

import json
from pathlib import Path
from datasets import load_dataset
import random

# =========================================================
# CONFIG (LOCK THIS FOR ALL MODELS / ALL RUNS)
# =========================================================
DATA_JSONL = "Dataset/VIOLIN_12-10-2025/final/llm_adjuvant_evidence_corpus_finetune.jsonl"
OUT_JSON   = "Dataset/VIOLIN_12-10-2025/final/splits_fixed.json"

TEST_RATIO = 0.10   # stable final reporting set
VAL_RATIO  = 0.05   # smaller validation set for selection
SEED = 42

# =========================================================
# LOAD DATA
# =========================================================
ds = load_dataset("json", data_files={"data": DATA_JSONL}, split="data")
pmids = sorted({row["pmid"] for row in ds})

# =========================================================
# DETERMINISTIC SHUFFLE
# =========================================================
rng = random.Random(SEED)
rng.shuffle(pmids)

# =========================================================
# SPLIT
# =========================================================
n_total = len(pmids)

n_test = max(1, int(n_total * TEST_RATIO))
test_pmids = pmids[:n_test]

train_val_pmids = pmids[n_test:]
n_val = max(1, int(len(train_val_pmids) * VAL_RATIO))
val_pmids = train_val_pmids[:n_val]

train_pmids = train_val_pmids[n_val:]

# =========================================================
# SAVE
# =========================================================
out = {
    "seed": SEED,
    "test_ratio": TEST_RATIO,
    "val_ratio": VAL_RATIO,
    "n_total": n_total,
    "n_train": len(train_pmids),
    "n_val": len(val_pmids),
    "n_test": len(test_pmids),
    "train_pmids": train_pmids,
    "val_pmids": val_pmids,
    "test_pmids": test_pmids,
}

Path(OUT_JSON).parent.mkdir(parents=True, exist_ok=True)
Path(OUT_JSON).write_text(json.dumps(out, indent=2), encoding="utf-8")

print("Saved:", OUT_JSON)
print(f"Total: {n_total}")
print(f"Train: {len(train_pmids)}")
print(f"Val:   {len(val_pmids)}")
print(f"Test:  {len(test_pmids)}")


Saved: Dataset/VIOLIN_12-10-2025/final/splits_fixed.json
Total: 298
Train: 256
Val:   13
Test:  29
